# Lewisham · address fields: `site_name`, `site_number`, `street_name`, `locality`, `postcode`

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

The five web columns are looked at together. Brent's conclusions to test here
(`semantic/brent/address.yaml`, `findings/brent/site_name.ipynb` and the other four):
`site_name` is the main address (a full address on its own for 84% of Brent records), the
other four fields are a fallback when it is a bare number, one part or missing, and
`site_number` arrives as mixed JSON types. Brent's rule "every record is in Brent" relied
on the ONS boundary check, which is not approved for Lewisham.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls, blanks and types, all five fields

Fetch the five fields for every Lewisham record (paged with `search_after`, sorted by
`id`), plus the four fields of the empty-record rule, so the 1,151 empty records can be
shown apart. For each field count null, empty or blank, and filled values by Python type
(aggregations hide the JSON type).

In [2]:
FIELDS = ["site_name", "site_number", "street_name", "locality", "postcode"]
RULE = ["status", "decision", "decision_date", "description"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", *FIELDS, *RULE, "bo_system"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

addr = pd.DataFrame(rows).reindex(columns=["id", *FIELDS, *RULE, "bo_system"])


def is_empty(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip() == "")


addr["empty_record"] = pd.concat([is_empty(addr[f]) for f in RULE], axis=1).all(axis=1)
print("Lewisham records:", len(addr), "| empty records:", int(addr["empty_record"].sum()))


def python_type(v: object) -> str:
    return "null" if v is None or v != v else type(v).__name__


table = []
for f in FIELDS:
    v = addr[f]
    blank = v.notna() & (v.astype(str).str.strip() == "")
    types = v[v.notna() & ~blank].map(python_type).value_counts().to_dict()
    table.append({"field": f, "null": int(v.isna().sum()), "empty or blank": int(blank.sum()),
                  "filled": int((v.notna() & ~blank).sum()), "filled %": round(100 * (v.notna() & ~blank).mean(), 2),
                  "str": types.get("str", 0), "int": types.get("int", 0), "float": types.get("float", 0),
                  "filled, kept records": int((v.notna() & ~blank & ~addr["empty_record"]).sum())})
pd.DataFrame(table)

Lewisham records: 29021 | empty records: 1151


,field,null,empty or blank,filled,filled %,str,int,float,"filled, kept records"
0,site_name,50,0,28971,99.83,28971,0,0,27820
1,site_number,22967,0,6054,20.86,4486,1568,0,6054
2,street_name,24638,3,4380,15.09,4380,0,0,4380
3,locality,28453,0,568,1.96,568,0,0,568
4,postcode,23413,2,5606,19.32,5606,0,0,4506


### Findings · Step 1

Live API, 2026-11-11, 29,021 Lewisham records (1,151 empty, 27,870 kept):

| field | null | blank | filled | filled % | str | int |
|---|---|---|---|---|---|---|
| site_name | 50 | 0 | 28,971 | 99.83% | 28,971 | 0 |
| site_number | 22,967 | 0 | 6,054 | 20.86% | 4,486 | **1,568** |
| street_name | 24,638 | 3 | 4,380 | 15.09% | 4,380 | 0 |
| locality | 28,453 | 0 | 568 | 1.96% | 568 | 0 |
| postcode | 23,413 | 2 | 5,606 | 19.32% | 5,606 | 0 |

- **`site_name` is almost always there** (99.8%), as in Brent. The 50 without one are all
  kept records (every empty record has a site_name).
- **`site_number` has mixed JSON types, as in Brent:** 1,568 numbers and 4,486 strings.
  Brent's rule (load as text) is needed here too.
- **The other four fields are filled more often than in Brent** (site_number 20.9% vs
  15.6%, street_name 15.1% vs 9.8%, postcode 19.3% vs 10.9%); locality less (2.0% vs 2.8%).
- **Empty records carry postcodes:** 1,100 of the 5,606 postcodes are on empty records;
  none of their site_number, street_name or locality is filled.
- 3 blank street_name and 2 blank postcode values (become null at load, `blank_to_null`).

## Step 2 · Brent's six kinds of `site_name`, and whether the fallback is there

Kept records only (empty records left out). Each `site_name` is tidied and sorted with the
service's own code (`address_kind` in `pipeline/clean.py`, the derived rule `address` in
Brent's `cleaning.yaml`), so the kinds are exactly Brent's. For each kind: how many records
also have a street_name, a postcode or a site_number.

In [3]:
from london_brent_lewisham_chat.pipeline.clean import address_kind, tidy_text


def tidy(v: object) -> object:
    # pandas gives a missing value as NaN; the service's code expects None.
    return None if v is None or v != v else tidy_text(v)


kept = addr[~addr["empty_record"]].copy()
kept["kind"] = [address_kind(tidy(v)) for v in kept["site_name"]]
for f in FIELDS:
    kept[f] = [tidy(v) for v in kept[f]]
KINDS = ["with_full_postcode", "several_parts", "number_only", "one_part", "street_record", "missing"]
kinds = kept.groupby("kind").agg(count=("id", "size"),
                                  has_street_name=("street_name", lambda s: int(s.notna().sum())),
                                  has_postcode=("postcode", lambda s: int(s.notna().sum())),
                                  has_site_number=("site_number", lambda s: int(s.notna().sum()))).reindex(KINDS).fillna(0).astype(int)
kinds.insert(1, "share_%", (100 * kinds["count"] / len(kept)).round(2))
print("Kept records:", len(kept))
kinds

Kept records: 27870


,count,share_%,has_street_name,has_postcode,has_site_number
kind,,,,,
with_full_postcode,22509,80.76,1,126,1705
several_parts,1568,5.63,602,603,624
number_only,2838,10.18,2838,2838,2832
one_part,899,3.23,890,890,882
street_record,6,0.02,0,0,0
missing,50,0.18,49,49,11


In [4]:
# Examples of each kind (most common site_name values).
for k in KINDS:
    top = kept.loc[kept["kind"] == k, "site_name"].value_counts().head(6)
    print(k, "->", top.to_dict() if len(top) else "(none)")

with_full_postcode -> {'CARPETRIGHT, LOAMPIT VALE, LONDON, SE13 7SN': 61, 'CONVOYS LTD, PRINCE STREET, LONDON, SE8 3JH': 60, "164-196 TRUNDLEY'S ROAD AND 1-9 SANFORD STREET, LONDON, SE8 5JE": 58, "FORMER TESCO'S CAR PARK, 209 CONINGTON ROAD, LONDON, SE13 7LH": 45, 'NEPTUNE WORKS, CANAL APPROACH, LONDON, SE8 5JF': 45, '152A LEE HIGH ROAD, LONDON, SE13 5PR': 41}
several_parts -> {'LEWISHAM GATEWAY, LEWISHAM HIGH STREET, LONDON, SE13': 65, 'CROWN AND NEW BALTIC, PARK AND BRIDGE AND VICTORIA WHARVES, BOUNDED BY GROVE STREET,  DRAGOON STREET AND EVELYN STREET, OXESTALLS ROAD, LONDON, SE8': 58, 'EARL PUMPING STATION, CHILTON GROVE, LONDON, SE8': 38, 'DEPTFORD CHURCH STREET SITE, THAMES TUNNEL, LONDON': 37, 'SYDENHAM HILL ESTATE, SYDENHAM HILL, LONDON, SE26': 33, 'LAND TO THE EAST OF THE RAILWAY, BECKENHAM PLACE PARK, BECKENHAM, SE6': 29}
number_only -> {'1': 63, '2': 59, '9': 35, '14': 33, '10': 31, '7': 31}
one_part -> {'LAND BOUNDED BY': 4, 'LEWISHAM HOSPITAL': 4, 'Moore House': 4, 'FORMER

### Findings · Step 2

Live API, 2026-11-11, 27,870 kept records (Brent's shares from its Step 5 for comparison):

| Kind of `site_name` | Count | Share | Brent | Has `street_name` | Has `postcode` |
|---|---|---|---|---|---|
| with full postcode | 22,509 | 80.76% | 84.10% | 1 | 126 |
| several parts, no postcode | 1,568 | 5.63% | 6.98% | 602 | 603 |
| number only | 2,838 | 10.18% | 4.33% | 2,838 | 2,838 |
| one part, no postcode | 899 | 3.23% | 3.35% | 890 | 890 |
| street record | 6 | 0.02% | 1.12% | 0 | 0 |
| missing | 50 | 0.18% | 0.13% | 49 | 49 |

- **Brent's conclusion holds:** 81% of site names are a full address on their own, and
  then the other fields are almost never filled.
- **The fallback is there when needed:** number only (2,838), one part (899) and missing
  (50) are 3,787 records (13.6%); 3,777 of them have a street_name and a postcode. Number-only
  site names are more than twice as common as in Brent.
- **Several parts, no postcode (1,568):** mostly large sites described in words ("Lewisham
  Gateway, Lewisham High Street, London, SE13", "Earl Pumping Station, Chilton Grove, SE8"),
  often with only the postcode district; 965 have no fallback, as in Brent.
- **Street records are rare** (6, against 408 in Brent).
- **The same site repeats:** e.g. Carpetright, Loampit Vale (61), Convoys Ltd, Prince Street
  (60), so as in Brent the same site_name means the same site.
- So Brent's derived `address` (site_name, completed from the other four fields when it
  is a number only, one part or missing) works for Lewisham unchanged.

## Step 3 · Values of the four fallback fields

Kept records, tidied (Step 2). For each field: distinct values and the most common ones.
Then field-specific checks, as in Brent's notebooks:
- `site_number`: digits only vs other text, and the same number as both a JSON number and
  a string;
- `street_name`: case variants of the same street;
- `locality`: leading/trailing commas, case variants;
- `postcode`: full UK postcode, district only, or something else.

In [5]:
for f in ["site_number", "street_name", "locality", "postcode"]:
    v = kept[f].dropna()
    print(f"{f}: {len(v)} filled, {v.astype(str).nunique()} distinct")
    print("  most common:", v.astype(str).value_counts().head(10).to_dict())
    print()

site_number: 6054 filled, 2276 distinct
  most common: {'1': 116, '2': 88, '10': 58, '6': 58, '7': 58, '14': 54, '4': 54, '3': 52, '9': 52, '13': 48}

street_name: 4380 filled, 1189 distinct
  most common: {'NEW CROSS ROAD': 94, 'New Cross Road': 70, 'LEWISHAM HIGH STREET': 64, 'Stanstead Road': 49, 'Deptford High Street': 49, 'Brockley Road': 47, 'Lewisham High Street': 44, 'Bromley Road': 44, 'Sydenham Road': 42, 'SYDENHAM ROAD': 40}

locality: 568 filled, 29 distinct
  most common: {'LONDON': 199, 'London': 36, 'Brockley': 35, 'Lewisham Central': 29, 'Rushey Green': 28, 'New Cross': 23, 'Telegraph Hill': 23, 'Forest Hill': 19, 'Blackheath': 18, 'Ladywell': 18}

postcode: 4506 filled, 1453 distinct
  most common: {'SE13': 191, 'SE8': 157, 'SE6': 139, 'SE14': 126, 'SE23': 108, 'SE4': 107, 'SE26': 99, 'SE12': 70, 'SE14 6TA': 25, 'SE14 5BA': 20}



In [6]:
# site_number: kinds of value, and numbers that arrive both as JSON number and string.
sn = kept["site_number"].dropna()
kind = sn.map(lambda v: "JSON number" if isinstance(v, (int, float)) else
              ("string, digits only" if str(v).isdigit() else "string, other text"))
print(kind.value_counts().to_dict())
as_number = set(sn[kind == "JSON number"].map(lambda v: str(int(v))))
as_string = set(sn[kind == "string, digits only"].astype(str))
print("Numbers seen both ways (e.g. 2 and '2'):", len(as_number & as_string))
print("Other text, most common:", sn[kind == "string, other text"].astype(str).value_counts().head(15).to_dict())

{'string, other text': 2243, 'string, digits only': 2243, 'JSON number': 1568}
Numbers seen both ways (e.g. 2 and '2'): 224
Other text, most common: {'1a': 10, '7-9': 8, '65-71': 7, '1-3': 6, '2-4': 5, '16-18': 5, '173-175': 5, '74-76': 5, '72-78': 5, '81-85': 5, 'LAND BOUNDED BY': 4, '20-22': 4, '177-181': 4, 'LEWISHAM HOSPITAL': 4, '30-32': 4}


In [7]:
# street_name: the same street in different case.
st = kept["street_name"].dropna().astype(str)
by_upper = st.groupby(st.str.upper()).agg(lambda s: sorted(set(s)))
variants = by_upper[by_upper.map(len) > 1]
print("Streets written in more than one case:", len(variants), "| e.g.", variants.head(5).to_dict())

# locality: every value (few), to see commas and case.
print()
print("locality, all values:", kept["locality"].dropna().astype(str).value_counts().to_dict())

Streets written in more than one case: 360 | e.g. {'ABERDEEN TERRACE': ['ABERDEEN TERRACE', 'Aberdeen Terrace'], 'ABERNETHY ROAD': ['ABERNETHY ROAD', 'Abernethy Road'], 'ADAMSRILL ROAD': ['ADAMSRILL ROAD', 'Adamsrill Road'], 'ADDINGTON GROVE': ['ADDINGTON GROVE', 'Addington Grove'], 'ADELAIDE AVENUE': ['ADELAIDE AVENUE', 'Adelaide Avenue']}

locality, all values: {'LONDON': 199, 'London': 36, 'Brockley': 35, 'Lewisham Central': 29, 'Rushey Green': 28, 'New Cross': 23, 'Telegraph Hill': 23, 'Forest Hill': 19, 'Blackheath': 18, 'Ladywell': 18, 'Crofton Park': 17, 'Sydenham': 15, 'Catford South': 14, 'Perry Vale': 13, 'Lee Green': 12, 'Bellingham': 11, 'Evelyn': 11, 'Downham': 9, 'Whitefoot': 8, 'Grove Park': 7, 'NULL': 6, 'Bromley': 4, 'BROMLEY': 4, 'Catford': 3, 'Hither Green': 2, 'Lewisham': 1, 'GROVE PARK': 1, 'LONDO': 1, 'FOREST HILL': 1}


In [8]:
# postcode: full, district only, or other.
import re
FULL = re.compile(r"^[A-Z]{1,2}\d[A-Z\d]? ?\d[A-Z]{2}$")
DISTRICT = re.compile(r"^[A-Z]{1,2}\d[A-Z\d]?$")
pc = kept["postcode"].dropna().astype(str).str.upper()
pc_kind = pc.map(lambda v: "full" if FULL.match(v) else ("district only" if DISTRICT.match(v) else "other"))
print(pc_kind.value_counts().to_dict())
print("Other:", pc[pc_kind == "other"].value_counts().head(15).to_dict())
print("Districts (first part), most common:", pc.str.split().str[0].value_counts().head(15).to_dict())

{'full': 3418, 'district only': 1071, 'other': 17}
Other: {'SE13 5L': 2, 'SE23 2H': 1, 'SE6 4E': 1, 'SE13 RY': 1, 'SE23 DD': 1, 'SE8 4B': 1, 'SE4XY': 1, 'SE13 RT': 1, 'SE14 67P': 1, 'SE8 4Q': 1, '13 6TS': 1, 'SE6 746': 1, 'SE12 8A': 1, 'SE23 3E': 1, 'SE26 4L': 1}
Districts (first part), most common: {'SE13': 863, 'SE6': 766, 'SE23': 617, 'SE14': 511, 'SE4': 462, 'SE8': 408, 'SE26': 402, 'SE12': 233, 'BR1': 87, 'SE3': 65, 'SE16': 28, 'SE15': 23, 'SE10': 17, 'SE9': 16, 'BR3': 2}


### Findings · Step 3

**`site_number`** (6,054 filled, 2,276 distinct):
- **Mixed JSON types, as in Brent:** 1,568 JSON numbers, 2,243 digit strings, 2,243 other
  text. 224 numbers arrive both ways (`2` and `'2'`). Loading as text (Brent's
  `number_as_text`) is needed.
- Other text: letters (`1a`), ranges (`7-9`, `65-71`, `173-175`), names (`LAND BOUNDED BY`,
  `LEWISHAM HOSPITAL`).

**`street_name`** (4,380 filled, 1,189 distinct):
- Real street names, led by Lewisham's main roads (New Cross Road, Lewisham High Street,
  Stanstead Road, Deptford High Street).
- **Case varies, as in Brent:** 360 streets appear in more than one case (`NEW CROSS ROAD`
  94 / `New Cross Road` 70).

**`locality`** (568 filled, 29 distinct):
- Mostly London (`LONDON` 199, `London` 36) and **Lewisham ward names** (Brockley, Lewisham
  Central, Rushey Green, Telegraph Hill, …).
- **`NULL` (6) is the text "NULL",** not a missing value.
- `LONDO` (1) is cut off; `Bromley` / `BROMLEY` (8) is a postal town.
- No leading/trailing commas (Brent had `London,`).

**`postcode`** (4,506 filled, 1,453 distinct):
- 3,418 full, 1,071 district only (`SE13`, `SE8`, …), **17 malformed** (`SE13 5L`, `SE4XY`,
  `SE14 67P`, `13 6TS`, …).
- Districts are mostly SE (SE13, SE6, SE23, SE14, SE4, SE8, SE26, SE12); also BR1 (87),
  SE3, SE16, SE15, SE10, SE9, BR3. Postal districts cross borough lines, so a district
  alone does not show the borough (out-of-borough notebook, Step 1).